In [7]:
import urllib.request


ANSWERS_URL = "https://raw.githubusercontent.com/pythonmcpi/wordle-wordlist/main/answerlist.txt"
GUESSES_URL = "https://raw.githubusercontent.com/pythonmcpi/wordle-wordlist/main/wordlist.txt"


def load_words_from_url(url: str) -> list[str]:
    with urllib.request.urlopen(url) as response:
        lines = response.read().decode("utf-8").splitlines()

    return sorted(set(
        word.strip().lower()
        for word in lines
        if len(word.strip()) == 5 and word.strip().isalpha()
    ))


answers = load_words_from_url(ANSWERS_URL)
guesses = load_words_from_url(GUESSES_URL)

# Make sure every possible answer is also allowed as a guess.
guesses = sorted(set(guesses) | set(answers))

print("answers:", len(answers))
print("guesses:", len(guesses))

answers: 2309
guesses: 12947


In [8]:
import csv
import json
import math
import os
import hashlib
from collections import Counter
from statistics import mean

import numpy as np


PATTERN_COUNT = 243
GREEN_ID = 242  # (2, 2, 2, 2, 2) encoded in base 3
REFERENCE_LAMBDAS = (0.0, 0.25, 0.5, 0.75, 1.0)


def get_pattern_tuple(answer: str, guess: str) -> tuple[int, int, int, int, int]:
    """
    Return Wordle feedback pattern.

    0 = gray
    1 = yellow
    2 = green

    Duplicate letters are handled using Wordle-style rules:
    greens are consumed first, then yellows are assigned only when
    the answer has remaining copies of that letter.
    """

    pattern = [0] * 5
    remaining = Counter()

    # First pass: mark greens and count unmatched answer letters.
    for i in range(5):
        if guess[i] == answer[i]:
            pattern[i] = 2
        else:
            remaining[answer[i]] += 1

    # Second pass: mark yellows using remaining letter counts.
    for i in range(5):
        if pattern[i] == 2:
            continue

        ch = guess[i]

        if remaining[ch] > 0:
            pattern[i] = 1
            remaining[ch] -= 1

    return tuple(pattern)


def encode_pattern(pattern: tuple[int, int, int, int, int]) -> int:
    """
    Encode a pattern tuple into an integer from 0 to 242.
    """

    return sum(pattern[i] * (3 ** i) for i in range(5))


def decode_pattern(pattern_id: int) -> tuple[int, int, int, int, int]:
    """
    Decode a pattern ID back into a tuple.
    """

    values = []

    for _ in range(5):
        values.append(pattern_id % 3)
        pattern_id //= 3

    return tuple(values)


def build_pattern_table(answers: list[str], guesses: list[str]) -> np.ndarray:
    """
    Build pattern_table[answer_index, guess_index] = pattern_id.

    dtype uint8 is enough because pattern_id is between 0 and 242.
    """

    table = np.empty((len(answers), len(guesses)), dtype=np.uint8)

    for i, answer in enumerate(answers):
        for j, guess in enumerate(guesses):
            table[i, j] = encode_pattern(get_pattern_tuple(answer, guess))

    return table


def hash_words(words: list[str]) -> str:
    """
    Return a stable hash for a word list.
    """

    text = "\n".join(words)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_words(path: str, words: list[str]) -> None:
    """
    Save a word list as one word per line.
    """

    with open(path, "w", encoding="utf-8") as f:
        for word in words:
            f.write(word + "\n")


def get_partition_counts_from_array(
    candidate_array: np.ndarray,
    guess_index: int,
) -> np.ndarray:
    """
    Return counts of feedback patterns for a guess over current candidates.
    """

    pattern_ids = pattern_table[candidate_array, guess_index]

    return np.bincount(
        pattern_ids,
        minlength=PATTERN_COUNT,
    )


def modified_score_from_components(
    block_count_score: float,
    uniformity_score: float,
    lam: float,
) -> float:
    """
    Compute Modified Entropy Maximizing score.

    This score is centered at standard entropy:

        lam = 0.5 -> standard normalized entropy
        lam > 0.5 -> more weight on number of blocks
        lam < 0.5 -> more weight on uniformity

    score =
        block_count_score ** (2 * lam)
        * uniformity_score ** (2 * (1 - lam))
    """

    if block_count_score <= 0 or uniformity_score <= 0:
        return 0.0

    return (
        block_count_score ** (2 * lam)
        * uniformity_score ** (2 * (1 - lam))
    )


def metrics_from_counts(
    counts: np.ndarray,
    n_candidates: int,
    lam: float,
) -> dict:
    """
    Compute partition metrics from pattern counts.
    """

    if not 0 <= lam <= 1:
        raise ValueError("lam must be between 0 and 1.")

    sizes = counts[counts > 0].astype(np.float64)
    k = len(sizes)

    if k == 0:
        raise ValueError("No nonempty partition blocks.")

    max_block = int(sizes.max())
    expected_remaining = float(np.sum(sizes * sizes) / n_candidates)

    if k <= 1:
        entropy = 0.0
        block_count_score = 0.0
        uniformity_score = 0.0
        normalized_entropy = 0.0
        score = 0.0
    else:
        probs = sizes / n_candidates

        entropy = float(-np.sum(
            probs * np.log2(probs)
        ))

        max_possible_blocks = min(PATTERN_COUNT, n_candidates)

        block_count_score = math.log2(k) / math.log2(max_possible_blocks)
        uniformity_score = entropy / math.log2(k)
        normalized_entropy = entropy / math.log2(max_possible_blocks)

        score = modified_score_from_components(
            block_count_score=block_count_score,
            uniformity_score=uniformity_score,
            lam=lam,
        )

    return {
        "score": score,
        "lambda": lam,
        "n_blocks": int(k),
        "entropy": entropy,
        "normalized_entropy": normalized_entropy,
        "block_count_score": block_count_score,
        "uniformity_score": uniformity_score,
        "max_block": max_block,
        "expected_remaining": expected_remaining,
    }


def group_candidate_indices_by_pattern(
    candidate_indices: tuple[int, ...],
    candidate_array: np.ndarray,
    guess_index: int,
) -> dict[int, tuple[int, ...]]:
    """
    Split candidate answer indices by pattern ID using the precomputed table.
    """

    pattern_ids = pattern_table[candidate_array, guess_index]

    groups = {}

    for answer_index, pattern_id in zip(candidate_indices, pattern_ids):
        pattern_id = int(pattern_id)

        if pattern_id not in groups:
            groups[pattern_id] = []

        groups[pattern_id].append(answer_index)

    return {
        pattern_id: tuple(indices)
        for pattern_id, indices in groups.items()
    }


def pick_best_guess_modified_entropy(
    candidate_indices: tuple[int, ...],
    guess_indices: tuple[int, ...],
    lam: float,
) -> dict:
    """
    Pick the best guess using Modified Entropy Maximizing.

    Tie-breaking:
    1. Higher modified score
    2. Lower expected remaining
    3. Lower max block
    4. Prefer guesses that are still possible answers
    5. Lexicographic order
    """

    candidate_array = np.fromiter(
        candidate_indices,
        dtype=np.int32,
        count=len(candidate_indices),
    )

    candidate_set = set(candidate_indices)

    best_guess_index = None
    best_metrics = None
    best_key = None

    for guess_index in guess_indices:
        counts = get_partition_counts_from_array(
            candidate_array=candidate_array,
            guess_index=guess_index,
        )

        metrics = metrics_from_counts(
            counts=counts,
            n_candidates=len(candidate_indices),
            lam=lam,
        )

        answer_index_for_guess = guess_index_to_answer_index[guess_index]

        is_candidate = (
            1
            if answer_index_for_guess is not None
            and answer_index_for_guess in candidate_set
            else 0
        )

        key = (
            metrics["score"],
            -metrics["expected_remaining"],
            -metrics["max_block"],
            is_candidate,
            -guess_index,
        )

        if best_key is None or key > best_key:
            best_key = key
            best_guess_index = guess_index
            best_metrics = metrics

    groups = group_candidate_indices_by_pattern(
        candidate_indices=candidate_indices,
        candidate_array=candidate_array,
        guess_index=best_guess_index,
    )

    return {
        "guess_index": best_guess_index,
        "guess": guesses[best_guess_index],
        "groups": groups,
        **best_metrics,
    }


def get_guess_indices_for_state(
    candidate_indices: tuple[int, ...],
    all_guess_indices: tuple[int, ...],
    guess_pool_mode: str = "all",
    hybrid_threshold: int = 20,
) -> tuple[int, ...]:
    """
    Select the guess pool for a state.

    guess_pool_mode:
    - "all": use every allowed guess.
    - "candidates": only use current possible answers as guesses.
    - "hybrid": use all guesses for large states, candidates only for small states.
    """

    if guess_pool_mode == "all":
        return all_guess_indices

    if guess_pool_mode == "candidates":
        return tuple(
            guess_to_index[answers[i]]
            for i in candidate_indices
        )

    if guess_pool_mode == "hybrid":
        if len(candidate_indices) >= hybrid_threshold:
            return all_guess_indices

        return tuple(
            guess_to_index[answers[i]]
            for i in candidate_indices
        )

    raise ValueError("guess_pool_mode must be one of: all, candidates, hybrid")


def evaluate_modified_entropy(
    answers: list[str],
    guesses: list[str],
    lam: float = 0.5,
    first_guess: str | None = None,
    guess_pool_mode: str = "all",
    hybrid_threshold: int = 20,
) -> dict:
    """
    Evaluate Modified Entropy Maximizing.

    Strategy:
    - If first_guess is None, choose the first guess by Modified Entropy Maximizing.
    - If first_guess is given, use it first.
    - After that, choose each next guess by Modified Entropy Maximizing.
    - Return actual Wordle-style chains and attempt counts.

    lam:
    - 0.5 is standard entropy maximizing.
    - Larger values emphasize the number of partition blocks.
    - Smaller values emphasize uniformity.
    """

    if not 0 <= lam <= 1:
        raise ValueError("lam must be between 0 and 1.")

    answer_indices = tuple(range(len(answers)))
    all_guess_indices = tuple(range(len(guesses)))

    if first_guess is not None:
        first_guess = first_guess.lower()

        if first_guess not in guess_to_index:
            raise ValueError(f"{first_guess!r} is not in the allowed guess list.")

    memo = {}
    policy = {}

    def solve_state(candidate_indices: tuple[int, ...]) -> dict[int, tuple[int, ...]]:
        """
        Return solving chains from the current candidate state.

        Keys:
            answer indices

        Values:
            tuple of guess indices forming the chain from this state
        """

        candidate_indices = tuple(sorted(candidate_indices))

        if candidate_indices in memo:
            return memo[candidate_indices]

        if len(candidate_indices) == 0:
            raise ValueError("Candidate state is empty.")

        if len(candidate_indices) == 1:
            answer_index = candidate_indices[0]
            final_guess_index = guess_to_index[answers[answer_index]]

            memo[candidate_indices] = {
                answer_index: (final_guess_index,)
            }

            return memo[candidate_indices]

        state_guess_indices = get_guess_indices_for_state(
            candidate_indices=candidate_indices,
            all_guess_indices=all_guess_indices,
            guess_pool_mode=guess_pool_mode,
            hybrid_threshold=hybrid_threshold,
        )

        best = pick_best_guess_modified_entropy(
            candidate_indices=candidate_indices,
            guess_indices=state_guess_indices,
            lam=lam,
        )

        guess_index = best["guess_index"]
        groups = best["groups"]

        policy[candidate_indices] = best

        chains = {}

        for pattern_id, subgroup in groups.items():
            if pattern_id == GREEN_ID:
                for answer_index in subgroup:
                    chains[answer_index] = (guess_index,)
                continue

            subgroup_chains = solve_state(subgroup)

            for answer_index, subchain in subgroup_chains.items():
                chains[answer_index] = (guess_index,) + subchain

        memo[candidate_indices] = chains
        return chains

    if first_guess is None:
        chains_by_answer_index = solve_state(answer_indices)
        actual_first_guess = policy[answer_indices]["guess"]
        first_guess_profile = policy[answer_indices]
    else:
        first_guess_index = guess_to_index[first_guess]
        candidate_array = np.fromiter(
            answer_indices,
            dtype=np.int32,
            count=len(answer_indices),
        )

        first_counts = get_partition_counts_from_array(
            candidate_array=candidate_array,
            guess_index=first_guess_index,
        )

        first_guess_profile = {
            "guess_index": first_guess_index,
            "guess": first_guess,
            **metrics_from_counts(
                counts=first_counts,
                n_candidates=len(answer_indices),
                lam=lam,
            ),
        }

        first_groups = group_candidate_indices_by_pattern(
            candidate_indices=answer_indices,
            candidate_array=candidate_array,
            guess_index=first_guess_index,
        )

        chains_by_answer_index = {}

        for pattern_id, subgroup in first_groups.items():
            if pattern_id == GREEN_ID:
                for answer_index in subgroup:
                    chains_by_answer_index[answer_index] = (first_guess_index,)
                continue

            subgroup_chains = solve_state(subgroup)

            for answer_index, subchain in subgroup_chains.items():
                chains_by_answer_index[answer_index] = (first_guess_index,) + subchain

        actual_first_guess = first_guess

    chains_by_answer = {
        answers[answer_index]: tuple(guesses[guess_index] for guess_index in chain)
        for answer_index, chain in chains_by_answer_index.items()
    }

    attempts_by_answer = {
        answer: len(chain)
        for answer, chain in chains_by_answer.items()
    }

    values = list(attempts_by_answer.values())

    return {
        "algorithm": "Modified Entropy Maximizing",
        "lambda": lam,
        "first_guess": actual_first_guess,
        "guess_pool_mode": guess_pool_mode,
        "mean": mean(values),
        "max": max(values),
        "results": attempts_by_answer,
        "chains": chains_by_answer,
        "first_guess_profile": first_guess_profile,
        "policy_state_count": len(policy),
        "memo_state_count": len(memo),
    }


def attempt_distribution(result: dict) -> list[dict]:
    """
    Return PMF of actual attempt counts.
    """

    counts = Counter(result["results"].values())
    total = sum(counts.values())

    return [
        {
            "attempts": attempts,
            "count": counts[attempts],
            "probability": counts[attempts] / total,
        }
        for attempts in sorted(counts)
    ]


def get_worst_answers(result: dict) -> list[str]:
    """
    Return answers that achieved the maximum number of attempts.
    """

    max_attempts = result["max"]

    return sorted(
        answer
        for answer, attempts in result["results"].items()
        if attempts == max_attempts
    )


def format_chain(chain: tuple[str, ...], answer: str) -> str:
    """
    Format a solving chain like A-B-C-D-E(correct).
    """

    words = list(chain)

    if len(words) == 0:
        return ""

    if words[-1] == answer:
        words[-1] = words[-1] + "(correct)"
    else:
        words.append(answer + "(correct)")

    return "-".join(words)


def build_reference_table_rows(
    reference_lambdas: tuple[float, ...] = REFERENCE_LAMBDAS,
) -> list[dict]:
    """
    Build a reference table for all first guesses over the full answer set.

    This table is independent of the recursive solving policy.
    It only measures the first-step partition induced by each guess.
    """

    candidate_array = np.arange(len(answers), dtype=np.int32)
    n_candidates = len(answers)

    rows = []

    for guess_index, guess in enumerate(guesses):
        counts = get_partition_counts_from_array(
            candidate_array=candidate_array,
            guess_index=guess_index,
        )

        base_metrics = metrics_from_counts(
            counts=counts,
            n_candidates=n_candidates,
            lam=0.5,
        )

        row = {
            "guess": guess,
            "n_blocks": base_metrics["n_blocks"],
            "entropy": base_metrics["entropy"],
            "normalized_entropy": base_metrics["normalized_entropy"],
            "block_count_score": base_metrics["block_count_score"],
            "uniformity_score": base_metrics["uniformity_score"],
            "max_block": base_metrics["max_block"],
            "expected_remaining": base_metrics["expected_remaining"],
        }

        for lam in reference_lambdas:
            key = f"score_lam_{lam:.2f}".replace(".", "_")
            row[key] = modified_score_from_components(
                block_count_score=base_metrics["block_count_score"],
                uniformity_score=base_metrics["uniformity_score"],
                lam=lam,
            )

        rows.append(row)

    entropy_key = "score_lam_0_50"

    rows.sort(key=lambda r: (
        -r[entropy_key],
        r["expected_remaining"],
        r["max_block"],
        r["guess"],
    ))

    for rank, row in enumerate(rows, start=1):
        row["rank_lam_0_50"] = rank

    return rows


def save_reference_table(rows: list[dict], path: str) -> None:
    """
    Save the reference partition table as CSV.
    """

    if len(rows) == 0:
        raise ValueError("No rows to save.")

    fieldnames = [
        "rank_lam_0_50",
        "guess",
        "n_blocks",
        "entropy",
        "normalized_entropy",
        "block_count_score",
        "uniformity_score",
        "max_block",
        "expected_remaining",
    ]

    for lam in REFERENCE_LAMBDAS:
        fieldnames.append(f"score_lam_{lam:.2f}".replace(".", "_"))

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=fieldnames,
            extrasaction="ignore",
        )

        writer.writeheader()
        writer.writerows(rows)

In [9]:
import os
import json
import hashlib
import numpy as np


OUTPUT_DIR = r"C:\Users\ws20w\OneDrive\Desktop\wordle"

os.makedirs(OUTPUT_DIR, exist_ok=True)

PATTERN_TABLE_PATH = os.path.join(OUTPUT_DIR, "pattern_table_uint8.npy")
ANSWERS_PATH = os.path.join(OUTPUT_DIR, "answers.txt")
GUESSES_PATH = os.path.join(OUTPUT_DIR, "guesses.txt")
METADATA_PATH = os.path.join(OUTPUT_DIR, "metadata.json")
REFERENCE_TABLE_PATH = os.path.join(OUTPUT_DIR, "wordle_partition_reference.csv")


def hash_words(words: list[str]) -> str:
    """
    Return a stable hash for a word list.
    """

    text = "\n".join(words)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def write_words(path: str, words: list[str]) -> None:
    """
    Save a word list as one word per line.
    """

    with open(path, "w", encoding="utf-8") as f:
        for word in words:
            f.write(word + "\n")


required_names = [
    "answers",
    "guesses",
    "build_pattern_table",
    "GREEN_ID",
    "build_reference_table_rows",
    "save_reference_table",
]

missing_names = [
    name
    for name in required_names
    if name not in globals()
]

if missing_names:
    raise NameError(
        "These names are not defined yet. "
        "Run cell 1 and cell 2 first: "
        + ", ".join(missing_names)
    )


current_metadata = {
    "answers_count": len(answers),
    "guesses_count": len(guesses),
    "answers_hash": hash_words(answers),
    "guesses_hash": hash_words(guesses),
    "pattern_encoding": "base3: p0 + 3*p1 + 9*p2 + 27*p3 + 81*p4",
    "green_id": GREEN_ID,
}


should_build_pattern_table = True

if os.path.exists(PATTERN_TABLE_PATH) and os.path.exists(METADATA_PATH):
    with open(METADATA_PATH, "r", encoding="utf-8") as f:
        saved_metadata = json.load(f)

    metadata_matches = (
        saved_metadata.get("answers_hash") == current_metadata["answers_hash"]
        and saved_metadata.get("guesses_hash") == current_metadata["guesses_hash"]
        and saved_metadata.get("answers_count") == current_metadata["answers_count"]
        and saved_metadata.get("guesses_count") == current_metadata["guesses_count"]
    )

    if metadata_matches:
        pattern_table = np.load(PATTERN_TABLE_PATH)

        if pattern_table.shape == (len(answers), len(guesses)):
            should_build_pattern_table = False
            print("loaded existing pattern table:", PATTERN_TABLE_PATH)
        else:
            print("saved pattern table shape mismatch; rebuilding")


if should_build_pattern_table:
    print("building pattern table")
    pattern_table = build_pattern_table(answers, guesses)

    np.save(PATTERN_TABLE_PATH, pattern_table)

    with open(METADATA_PATH, "w", encoding="utf-8") as f:
        json.dump(current_metadata, f, indent=2)

    print("saved pattern table:", PATTERN_TABLE_PATH)


write_words(ANSWERS_PATH, answers)
write_words(GUESSES_PATH, guesses)


answer_to_index = {
    word: i
    for i, word in enumerate(answers)
}

guess_to_index = {
    word: i
    for i, word in enumerate(guesses)
}

guess_index_to_answer_index = tuple(
    answer_to_index.get(guess)
    for guess in guesses
)


print("pattern table shape:", pattern_table.shape)
print("pattern table memory MB:", pattern_table.nbytes / 1024 / 1024)


print("building reference partition table")
reference_rows = build_reference_table_rows()
save_reference_table(reference_rows, REFERENCE_TABLE_PATH)


print("saved answers:", ANSWERS_PATH)
print("saved guesses:", GUESSES_PATH)
print("saved metadata:", METADATA_PATH)
print("saved reference table:", REFERENCE_TABLE_PATH)

print()
print("top 10 by standard entropy, lambda = 0.5:")

for row in reference_rows[:10]:
    print(
        f"{row['rank_lam_0_50']:2d}. {row['guess']} | "
        f"entropy={row['entropy']:.4f} | "
        f"blocks={row['n_blocks']} | "
        f"uniformity={row['uniformity_score']:.4f} | "
        f"expected={row['expected_remaining']:.2f} | "
        f"max_block={row['max_block']}"
    )

loaded existing pattern table: C:\Users\ws20w\OneDrive\Desktop\wordle\pattern_table_uint8.npy
pattern table shape: (2309, 12947)
pattern table memory MB: 28.50973415374756
building reference partition table
saved answers: C:\Users\ws20w\OneDrive\Desktop\wordle\answers.txt
saved guesses: C:\Users\ws20w\OneDrive\Desktop\wordle\guesses.txt
saved metadata: C:\Users\ws20w\OneDrive\Desktop\wordle\metadata.json
saved reference table: C:\Users\ws20w\OneDrive\Desktop\wordle\wordle_partition_reference.csv

top 10 by standard entropy, lambda = 0.5:
 1. soare | entropy=5.8852 | blocks=127 | uniformity=0.8421 | expected=62.06 | max_block=182
 2. roate | entropy=5.8849 | blocks=126 | uniformity=0.8434 | expected=60.13 | max_block=194
 3. raise | entropy=5.8783 | blocks=132 | uniformity=0.8345 | expected=60.74 | max_block=167
 4. reast | entropy=5.8677 | blocks=147 | uniformity=0.8150 | expected=71.43 | max_block=226
 5. raile | entropy=5.8652 | blocks=128 | uniformity=0.8379 | expected=61.19 | max_b

In [11]:
LAMBDA_VALUE = 1
FIRST_GUESS = None
# FIRST_GUESS = "slate"

GUESS_POOL_MODE = "all"
# GUESS_POOL_MODE = "candidates"
# GUESS_POOL_MODE = "hybrid"

HYBRID_THRESHOLD = 20

TRACE_ANSWER = None
# TRACE_ANSWER = "baton"

CHAIN_LIMIT = 10


result = evaluate_modified_entropy(
    answers=answers,
    guesses=guesses,
    lam=LAMBDA_VALUE,
    first_guess=FIRST_GUESS,
    guess_pool_mode=GUESS_POOL_MODE,
    hybrid_threshold=HYBRID_THRESHOLD,
)

print("algorithm:", result["algorithm"])
print("lambda:", result["lambda"])
print("first guess:", result["first_guess"])
print("guess pool mode:", result["guess_pool_mode"])
print("mean:", f"{result['mean']:.4f}")
print("max:", result["max"])
print("policy states:", result["policy_state_count"])
print("memo states:", result["memo_state_count"])

print()
print("first guess partition profile:")
profile = result["first_guess_profile"]

print("guess:", profile["guess"])
print("score:", f"{profile['score']:.6f}")
print("entropy:", f"{profile['entropy']:.6f}")
print("normalized entropy:", f"{profile['normalized_entropy']:.6f}")
print("blocks:", profile["n_blocks"])
print("block count score:", f"{profile['block_count_score']:.6f}")
print("uniformity score:", f"{profile['uniformity_score']:.6f}")
print("expected remaining:", f"{profile['expected_remaining']:.2f}")
print("max block:", profile["max_block"])

print()
print("attempt distribution:")
for row in attempt_distribution(result):
    print(
        f"{row['attempts']} attempts: "
        f"{row['count']} "
        f"({row['probability']:.2%})"
    )

print()

if TRACE_ANSWER is not None:
    TRACE_ANSWER = TRACE_ANSWER.lower()

    if TRACE_ANSWER not in result["chains"]:
        raise ValueError(f"{TRACE_ANSWER!r} is not in the answer list.")

    chain = result["chains"][TRACE_ANSWER]

    print("selected answer chain:")
    print(f"{TRACE_ANSWER}: {format_chain(chain, TRACE_ANSWER)}")

else:
    worst_answers = get_worst_answers(result)

    print("number of worst answers:", len(worst_answers))
    print("worst answer chains:")

    for answer in worst_answers[:CHAIN_LIMIT]:
        chain = result["chains"][answer]
        print(f"{answer}: {format_chain(chain, answer)}")   

algorithm: Modified Entropy Maximizing
lambda: 1
first guess: trace
guess pool mode: all
mean: 3.4305
max: 6
policy states: 525
memo states: 2468

first guess partition profile:
guess: trace
score: 0.832064
entropy: 5.830429
normalized entropy: 0.735718
blocks: 150
block count score: 0.912175
uniformity score: 0.806554
expected remaining: 73.95
max block: 246

attempt distribution:
1 attempts: 1 (0.04%)
2 attempts: 74 (3.20%)
3 attempts: 1232 (53.36%)
4 attempts: 935 (40.49%)
5 attempts: 66 (2.86%)
6 attempts: 1 (0.04%)

number of worst answers: 1
worst answer chains:
rover: trace-pines-dowly-rhomb-roger-rover(correct)


In [13]:
LAMBDA_VALUES = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

FIRST_GUESS = None
# FIRST_GUESS = "slate"

GUESS_POOL_MODE = "all"
# GUESS_POOL_MODE = "candidates"
# GUESS_POOL_MODE = "hybrid"

HYBRID_THRESHOLD = 20
CHAIN_LIMIT = 3


lambda_results = []

for lam in LAMBDA_VALUES:
    result = evaluate_modified_entropy(
        answers=answers,
        guesses=guesses,
        lam=lam,
        first_guess=FIRST_GUESS,
        guess_pool_mode=GUESS_POOL_MODE,
        hybrid_threshold=HYBRID_THRESHOLD,
    )

    worst_answers = get_worst_answers(result)

    row = {
        "lambda": lam,
        "first_guess": result["first_guess"],
        "mean": result["mean"],
        "max": result["max"],
        "policy_states": result["policy_state_count"],
        "memo_states": result["memo_state_count"],
        "worst_count": len(worst_answers),
        "worst_answers": worst_answers,
        "result": result,
    }

    lambda_results.append(row)

    print(
        f"lambda={lam:.2f} | "
        f"first={result['first_guess']} | "
        f"mean={result['mean']:.4f} | "
        f"max={result['max']} | "
        f"worst_count={len(worst_answers)}"
    )

    for answer in worst_answers[:CHAIN_LIMIT]:
        chain = result["chains"][answer]
        print("   ", f"{answer}: {format_chain(chain, answer)}")

    print()

lambda=0.00 | first=roate | mean=5.0645 | max=9 | worst_count=4
    cinch: roate-tatou-yarer-racer-areca-aback-ablow-cafes-cinch(correct)
    finch: roate-tatou-yarer-racer-areca-aback-ablow-cafes-finch(correct)
    pinch: roate-tatou-yarer-racer-areca-aback-ablow-cafes-pinch(correct)

lambda=0.10 | first=roate | mean=3.5219 | max=5 | worst_count=47
    craze: roate-bergs-franc-crave-craze(correct)
    drape: roate-bergs-franc-drake-drape(correct)
    fewer: roate-diels-funky-fever-fewer(correct)

lambda=0.20 | first=roate | mean=3.4937 | max=5 | worst_count=44
    batch: roate-taunt-clamp-ablow-batch(correct)
    broom: roate-grind-capos-brook-broom(correct)
    fewer: roate-diels-funky-fever-fewer(correct)

lambda=0.30 | first=roate | mean=3.4799 | max=5 | worst_count=42
    batch: roate-taunt-clamp-ablow-batch(correct)
    broom: roate-grind-capos-brook-broom(correct)
    cover: roate-mewls-chank-corer-cover(correct)

lambda=0.40 | first=roate | mean=3.4773 | max=5 | worst_count=44
